In [ ]:
!pip install unsloth

In [ ]:
import unsloth
import transformers

In [ ]:
from unsloth import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen2.5-VL-3B-Instruct-unsloth-bnb-4bit",
    load_in_4bit = True,
)

In [ ]:
import pandas as pd
from pathlib import Path
from PIL import Image
from sklearn.model_selection import train_test_split

DATA_DIR = Path("/kaggle/input/datasets/eazdanmostafarafin/bangla-meme-classification-dataset/preprocessed")
IMAGE_DIR = DATA_DIR / "Image_processed"
CSV_PATH = DATA_DIR / "Train_processed.csv"
OUTPUT_DIR = Path("/kaggle/working/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CSV_PATH)
df = df[["Image_name", "Target"]].dropna()
df = df[df["Image_name"].apply(lambda n: (IMAGE_DIR / n).is_file())].reset_index(drop=True)

LABELS = sorted(df["Target"].unique().tolist())
print(f"Classes: {LABELS}")
print(f"Total usable samples: {len(df)}")

train_df, temp_df = train_test_split(
    df, test_size=0.2, stratify=df["Target"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["Target"], random_state=42
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

In [ ]:


INSTRUCTION = (
    "You are an image classifier. Look at the image and classify it into exactly "
    f"one of these categories: {', '.join(LABELS)}. "
    "Respond with only the single category name."
)

def row_to_conversation(row):
    image = Image.open(IMAGE_DIR / row["Image_name"]).convert("RGB")
    return {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text",  "text": INSTRUCTION},
                ],
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": str(row["Target"])},
                ],
            },
        ]
    }

train_dataset = [row_to_conversation(r) for _, r in train_df.iterrows()]
val_dataset   = [row_to_conversation(r) for _, r in val_df.iterrows()]
print(f"Train convos: {len(train_dataset)} | Val convos: {len(val_dataset)}")



In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r              = 8,
    lora_alpha     = 16,
    lora_dropout   = 0,
    bias           = "none",
    random_state   = 42,
    use_rslora     = False,
    loftq_config   = None,
)

In [ ]:
from pathlib import Path
from unsloth import is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    train_dataset = train_dataset,
    eval_dataset  = val_dataset,
    args = SFTConfig(
        per_device_train_batch_size = 1,
        per_device_eval_batch_size  = 1,
        gradient_accumulation_steps = 8,
        warmup_steps                = 5,
        num_train_epochs            = 2,
        learning_rate               = 2e-4,
        fp16                        = not is_bf16_supported(),
        bf16                        = is_bf16_supported(),
        logging_steps               = 10,
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        lr_scheduler_type           = "linear",
        seed                        = 42,
        output_dir                  = str(OUTPUT_DIR),
        logging_dir                 = str(OUTPUT_DIR / "logs"),
        report_to                   = "none",
        remove_unused_columns       = False,
        dataset_text_field          = "",
        dataset_kwargs              = {"skip_prepare_dataset": True},
        dataset_num_proc            = 2,
        max_seq_length              = 2048,
        eval_strategy               = "epoch",
        save_strategy               = "steps",
        save_steps                  = 100,
        save_total_limit            = 3,
    ),
)

In [ ]:
import json

adapter_dir = OUTPUT_DIR / "final_adapter"
adapter_dir.mkdir(parents=True, exist_ok=True)

train_metrics = {}
eval_metrics = {}
training_completed = False

try:
    trainer_stats = trainer.train()
    print(trainer_stats)
    train_metrics = trainer_stats.metrics
    trainer.log_metrics("train", train_metrics)
    trainer.save_metrics("train", train_metrics)
    training_completed = True
except KeyboardInterrupt:
    print("Training interrupted. Saving current model state...")
finally:
    trainer.save_state()
    trainer.save_model(str(adapter_dir))
    tokenizer.save_pretrained(str(adapter_dir))
    print(f"Saved LoRA adapter + tokenizer to: {adapter_dir.resolve()}")

if training_completed:
    eval_metrics = trainer.evaluate()
    trainer.log_metrics("eval", eval_metrics)
    trainer.save_metrics("eval", eval_metrics)
else:
    interrupt_info = {
        "status": "interrupted",
        "global_step": int(trainer.state.global_step),
        "epoch": float(trainer.state.epoch) if trainer.state.epoch is not None else None,
    }
    with (OUTPUT_DIR / "interrupt_summary.json").open("w", encoding="utf-8") as f:
        json.dump(interrupt_info, f, indent=2)
    print(f"Saved interruption summary to: {(OUTPUT_DIR / 'interrupt_summary.json').resolve()}")

summary = {
    "train": train_metrics,
    "eval": eval_metrics,
    "training_completed": training_completed,
}
with (OUTPUT_DIR / "run_summary.json").open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"Saved trainer artifacts under: {OUTPUT_DIR.resolve()}")

# Optional: export a merged full model (larger disk usage).
try:
    merged_dir = OUTPUT_DIR / "final_merged_16bit"
    model.save_pretrained_merged(
        str(merged_dir),
        tokenizer,
        save_method = "merged_16bit",
    )
    print(f"Saved merged full model to: {merged_dir.resolve()}")
except Exception as e:
    print("Merged full-model export was skipped:", e)

In [ ]:
import json
from pathlib import Path
import pandas as pd
import torch
from tqdm import tqdm

OUTPUT_DIR = Path("outputs")
RESULTS_DIR = OUTPUT_DIR / "eval"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FastVisionModel.for_inference(model)

def predict_label(image, instruction):
    messages = [
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": instruction},
        ]},
    ]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(
        image,
        input_text,
        add_special_tokens = False,
        return_tensors     = "pt",
    ).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens = 16,
            do_sample      = False,
            temperature    = 0.0,
            use_cache      = True,
        )
    generated = output_ids[:, inputs["input_ids"].shape[1]:]
    return tokenizer.batch_decode(generated, skip_special_tokens=True)[0].strip()

def normalize_to_label(pred_text, labels):
    low = pred_text.lower()
    for lab in labels:
        if lab.lower() in low:
            return lab
    return pred_text

correct = 0
total   = 0
per_class_correct = {lab: 0 for lab in LABELS}
per_class_total   = {lab: 0 for lab in LABELS}
prediction_rows = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    image = Image.open(IMAGE_DIR / row["Image_name"]).convert("RGB")
    raw_pred = predict_label(image, INSTRUCTION)
    pred     = normalize_to_label(raw_pred, LABELS)
    true     = row["Target"]
    is_correct = int(pred == true)

    per_class_total[true] += 1
    total += 1
    if is_correct:
        correct += 1
        per_class_correct[true] += 1

    prediction_rows.append({
        "Image_name": row["Image_name"],
        "Target": true,
        "Prediction": pred,
        "Raw_Prediction": raw_pred,
        "Correct": is_correct,
    })

overall_acc = correct / total if total else 0.0
per_class_metrics = {}
for lab in LABELS:
    n = per_class_total[lab]
    acc = per_class_correct[lab] / n if n else 0.0
    per_class_metrics[lab] = {
        "correct": per_class_correct[lab],
        "total": n,
        "accuracy": acc,
    }

predictions_path = RESULTS_DIR / "test_predictions.csv"
metrics_path = RESULTS_DIR / "test_metrics.json"

pd.DataFrame(prediction_rows).to_csv(predictions_path, index=False)
with metrics_path.open("w", encoding="utf-8") as f:
    json.dump(
        {
            "overall": {
                "correct": correct,
                "total": total,
                "accuracy": overall_acc,
            },
            "per_class": per_class_metrics,
        },
        f,
        indent=2,
    )

print(f"\nTest accuracy: {correct}/{total} = {overall_acc:.4f}")
for lab in LABELS:
    m = per_class_metrics[lab]
    print(f"  {lab:10s}: {m['correct']}/{m['total']} = {m['accuracy']:.4f}")

print(f"Saved test predictions to: {predictions_path.resolve()}")
print(f"Saved test metrics to: {metrics_path.resolve()}")